In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('PGA_Data.csv')
df = df[["player", "season", "sg_ott", "sg_app", "sg_arg", "sg_putt", "sg_total"]]
display(df.head(5))

,player,season,sg_ott,sg_app,sg_arg,sg_putt,sg_total
0,Abraham Ancer,2022,0.86,-0.08,-0.13,0.20,0.85
1,Adam Hadwin,2022,0.18,0.31,0.75,0.36,1.60
2,Anirban Lahiri,2022,0.37,-1.09,0.74,-0.56,-0.54
3,Adam Long,2022,0.80,-0.02,-1.86,-1.46,-2.54
4,Alexander Noren,2022,0.19,-1.39,-0.36,0.53,-1.04


In [3]:
def normalize(series):
    # scale to 0-1 probability
    return (series - series.min()) / (series.max() - series.min())

df["p_tee_fairway"] = normalize(df["sg_ott"])
df["p_fairway_green"] = normalize(df["sg_app"])
df["p_rough_green"] = normalize(df["sg_app"] * 0.6)   
df["p_bunker_green"] = normalize(df["sg_arg"])
df["p_green_hole"] = normalize(df["sg_putt"])

In [4]:
p_tee_fairway   = df["p_tee_fairway"].mean()
p_fairway_green = df["p_fairway_green"].mean()
p_rough_green   = df["p_rough_green"].mean()
p_bunker_green  = df["p_bunker_green"].mean()
p_green_hole    = df["p_green_hole"].mean()

In [5]:
states = ["Tee", "Fairway", "Rough", "Bunker", "Green", "Hole"]
transition_matrix = pd.DataFrame(0.0, index=states, columns=states)

# Tee
transition_matrix.loc["Tee", "Fairway"] = p_tee_fairway
transition_matrix.loc["Tee", "Rough"]   = (1 - p_tee_fairway) * 0.7
transition_matrix.loc["Tee", "Bunker"]  = (1 - p_tee_fairway) * 0.3

# Fairway
# Fairway transitions
transition_matrix.loc["Fairway", "Green"]   = p_fairway_green
transition_matrix.loc["Fairway", "Rough"]   = (1 - p_fairway_green) * 0.5
transition_matrix.loc["Fairway", "Bunker"]  = (1 - p_fairway_green) * 0.3
transition_matrix.loc["Fairway", "Fairway"] = (1 - p_fairway_green) * 0.2 


# Rough
transition_matrix.loc["Rough", "Green"]  = p_rough_green
transition_matrix.loc["Rough", "Rough"]  = (1 - p_rough_green) * 0.5
transition_matrix.loc["Rough", "Bunker"] = (1 - p_rough_green) * 0.5

# Bunker
transition_matrix.loc["Bunker", "Green"] = p_bunker_green
transition_matrix.loc["Bunker", "Rough"] = 1 - p_bunker_green


# Green
transition_matrix.loc["Green", "Hole"]  = p_green_hole
transition_matrix.loc["Green", "Green"] = 1 - p_green_hole

# Hole is absorbing
transition_matrix.loc["Hole", "Hole"] = 1.0

print("Transition Matrix:\n")
print(transition_matrix.round(2))
print(type(transition_matrix))

Transition Matrix:

         Tee  Fairway  Rough  Bunker  Green  Hole
Tee      0.0     0.73   0.19    0.08   0.00  0.00
Fairway  0.0     0.07   0.17    0.10   0.66  0.00
Rough    0.0     0.00   0.17    0.17   0.66  0.00
Bunker   0.0     0.00   0.33    0.00   0.67  0.00
Green    0.0     0.00   0.00    0.00   0.44  0.56
Hole     0.0     0.00   0.00    0.00   0.00  1.00
<class 'pandas.core.frame.DataFrame'>


In [6]:
def simulate_hole(P, start_state=0):
    """
    Simulate one full hole using transition probabilities.
    P: transition probability matrix (numpy array)
    start_state: index of starting state (usually 0 = 'Tee')
    """
    current_state = start_state
    strokes = 0

    while current_state != len(P) - 1:  # Continue until 'Hole'
        strokes += 1
        current_state = np.random.choice(range(len(P)), p=P[current_state])

    return strokes


In [7]:
n_rounds = 1000  # Scale to many random samples
results = [simulate_hole(P) for _ in range(n_rounds)]

mean_strokes = np.mean(results)
variance_strokes = np.var(results)

print(f"Mean strokes per hole: {mean_strokes:.2f}")
print(f"Variance: {variance_strokes:.2f}")


NameError: name 'P' is not defined

In [ ]:
import matplotlib.pyplot as plt

plt.hist(results, bins=15, alpha=0.7, density=True, color='skyblue', edgecolor='black')
plt.title("Monte Carlo Simulation Across Entire Hole")
plt.xlabel("Total Strokes to Complete Hole")
plt.ylabel("Probability Density")
plt.show()
